In [2]:
# DASK client set

import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

# get path for path changes in Jupyter notebook: File - Open from Path - insert relative_path
notebook_path = os.path.abspath(".")
_, _, relative_path = notebook_path.partition('/all/')
relative_path = '/all/' + relative_path
relative_path

'/all/Model/CESM2/Earth_System_Predictability/DIC'

# Load modules

In [3]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.ticker as mticker
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.interpolate import griddata
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp


In [4]:
import xcesm
import gsw

In [5]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_NWP_preprocessing import CESM2_NWP_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

# Variable configuration

In [6]:

cfgs = {}
# varlist = ["DIC", "TEMP", "SALT"]
# varlist = ["DIC_ALT_CO2"]
varlist = ["TEMP"]

for varname in varlist:
    cfg = CESM2_NWP_config()
    cfg.year_s = 1954
    cfg.year_e = 2020
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')

# Read dataset

In [7]:
# lat_range = slice(23, 38)
# lon_range = slice(120, 180)
lat_range = slice(19, 40)
lon_range = slice(120, 180)

In [8]:
# define preprocessing function

# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]
# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'dz', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]


# exceptcv = [
#     'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
# ] + [cfg.var for cfg in cfgs.values()]
exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 
] + [cfg.var for cfg in cfgs.values()]


def process_coords_3d(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        # ds = ds.isel(z_t=slice(0, 39)) # ~39 layer (1000m)
        # ds = (ds.isel(z_t=slice(1, 39)) * ds.dz).sum(dim='z_t') / ds.dz.sum(dim='z_t')
        return ds
    else:
        return ds.set_coords(coord_vars)



def process_coords_3d_LE(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """
    Preprocessor function for CESM POP-style datasets.
    - Normalizes vertical coordinate: if z_t or z_t_2 exists, rename to 'depth'.
    - Replaces its values with z_t_new for consistency.
    - Optionally drops unnecessary coordinate variables for faster concatenation.
    """
    z_t_new = np.array([5.0000000e+00, 1.5000000e+01, 2.5000000e+01, 3.5000000e+01,
       4.5000000e+01, 5.5000000e+01, 6.5000000e+01, 7.5000000e+01,
       8.5000000e+01, 9.5000000e+01, 1.0500000e+02, 1.1500000e+02,
       1.2500000e+02, 1.3500000e+02, 1.4500000e+02, 1.5500000e+02,
       1.6509839e+02, 1.7547903e+02, 1.8629126e+02, 1.9766026e+02,
       2.0971138e+02, 2.2257828e+02, 2.3640883e+02, 2.5137015e+02,
       2.6765421e+02, 2.8548364e+02, 3.0511920e+02, 3.2686798e+02,
       3.5109348e+02, 3.7822760e+02, 4.0878464e+02, 4.4337769e+02,
       4.8273669e+02, 5.2772797e+02, 5.7937286e+02, 6.3886261e+02,
       7.0756329e+02, 7.8700250e+02, 8.7882520e+02, 9.8470581e+02,
       1.1062042e+03, 1.2445669e+03, 1.4004972e+03, 1.5739464e+03,
       1.7640033e+03, 1.9689442e+03, 2.1864565e+03, 2.4139714e+03,
       2.6490012e+03, 2.8893845e+03, 3.1334045e+03, 3.3797935e+03,
       3.6276702e+03, 3.8764519e+03, 4.1257681e+03, 4.3753926e+03,
       4.6251904e+03, 4.8750835e+03, 5.1250278e+03, 5.3750000e+03])
    
    # ------------------------------------------------------
    # 1️⃣ Normalize vertical coordinate name
    # ------------------------------------------------------
    if "z_t_2" in ds.dims:
        ds = ds.rename({"z_t_2": "depth"})
    elif "z_t" in ds.dims:
        ds = ds.rename({"z_t": "depth"})
    else:
        print("[Warning] No vertical coordinate (z_t or z_t_2) found — skipped.")
        return ds

    # Drop any leftover z_t/z_t_2 coordinate variable if it exists
    ds = ds.drop_vars(["z_t", "z_t_2"], errors="ignore")

    # ------------------------------------------------------
    # 2️⃣ Replace coordinate values with z_t_new
    # ------------------------------------------------------
    if "depth" in ds.coords:
        if len(ds["depth"]) == len(z_t_new):
            ds = ds.assign_coords(depth=z_t_new)
        else:
            print(f"[Warning] depth length mismatch: {len(ds['depth'])} vs {len(z_t_new)}")
    else:
        print("[Warning] depth coordinate missing after renaming.")

    # ------------------------------------------------------
    # 3️⃣ Clean up coordinate references inside variable attributes
    # ------------------------------------------------------
    for v in ds.data_vars:
        if "coordinates" in ds[v].attrs:
            ds[v].attrs["coordinates"] = (
                ds[v].attrs["coordinates"]
                .replace("z_t_2", "depth")
                .replace("z_t", "depth")
            )

    # ------------------------------------------------------
    # 4️⃣ Drop unnecessary coordinate variables and slice time
    # ------------------------------------------------------
    coord_vars = []
    for v in np.array(ds.coords):
        if v not in except_coord_vars:
            coord_vars.append(v)
    for v in np.array(ds.data_vars):
        if v not in except_coord_vars:
            coord_vars.append(v)

    if drop:
        ds = ds.drop(coord_vars, errors="ignore")
        ds = ds.sel(time=slice(sd, ed))
    else:
        ds = ds.set_coords(coord_vars)

    return ds



# exceptcv=['time','lon','lat','lev', 'TLONG', 'TLAT', 'z_t', cfg_var_DIC.var, cfg_var_Li_diat.var, cfg_var_Li_sp.var,
#          cfg_var_Vi_diat_Fe.var, cfg_var_Vi_diat_N.var, cfg_var_Vi_diat_P.var, cfg_var_Vi_diat_SiO3.var,
#          cfg_var_Vi_sp_Fe.var, cfg_var_Vi_sp_N.var, cfg_var_Vi_sp_P.var,
#          cfg_var_NPP_diat.var, cfg_var_NPP_sp.var]
# def process_coords(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
#     """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
#     coord_vars = []
#     for v in np.array(ds.coords) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     for v in np.array(ds.data_vars) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
    
#     if drop:
#         ds= ds.drop(coord_vars)
#         ds= ds.sel(time=slice(sd, ed))
#         return ds
#     else:
#         return ds.set_coords(coord_vars)

start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)


# ds = ds.isel(lev=slice(1, 11))

In [11]:
import xarray as xr
import numpy as np
import os
import glob
import time   # ← add

SIGMA0_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/sigma0"
MLD_ROOT    = "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD"

MLD_DELTA = 0.125


def compute_MLD_sigma(sig):
    z = sig.z_t
    rho0 = sig.isel(z_t=0)
    thresh = rho0 + MLD_DELTA

    exceed = sig >= thresh
    k = exceed.argmax("z_t")

    z_mld = z.isel(z_t=k)
    valid = exceed.any("z_t")
    z_mld = z_mld.where(valid)

    z_mld.name = "MLD"
    z_mld.attrs["long_name"] = "Mixed Layer Depth (sigma0 criterion)"
    z_mld.attrs["units"] = "cm"

    return z_mld


# --------------------------------------------------
# MAIN
# --------------------------------------------------
t_global0 = time.time()   # global timer

# for exp in ["ODA", "WDA", "LE"]:
for exp in ["LE"]:
    print(f"\n===== {exp} =====")

    exp_dir = os.path.join(SIGMA0_ROOT, exp)

    ens_dirs = sorted(
        [d for d in glob.glob(os.path.join(exp_dir, "*"))
         if os.path.isdir(d)]
    )

    # --------------------------------------------------
    # ensemble experiments
    # --------------------------------------------------
    if len(ens_dirs) > 0:

        for ens_path in ens_dirs:

            t_ens0 = time.time()   # ens timer

            ens = os.path.basename(ens_path)

            filelist = sorted(
                glob.glob(os.path.join(
                    ens_path,
                    f"sigma0_{exp}_{ens}_*.nc"
                ))
            )

            print(f"{exp} {ens}: {len(filelist)} files")

            out_dir = os.path.join(MLD_ROOT, exp, ens)
            os.makedirs(out_dir, exist_ok=True)

            printed = False
            n_done = 0

            for f in filelist:

                fname = os.path.basename(f)
                out_name = fname.replace("sigma0", "MLD")
                out_file = os.path.join(out_dir, out_name)

                if os.path.exists(out_file):
                    continue

                ds = xr.open_dataset(f)
                sig = ds["sigma0"]

                MLD = compute_MLD_sigma(sig)

                encoding = {
                    "MLD": {
                        "zlib": True,
                        "complevel": 4,
                        "dtype": "float32"
                    }
                }

                MLD.to_netcdf(out_file, encoding=encoding)

                ds.close()
                del ds, sig, MLD

                n_done += 1

                if not printed:
                    print("MLD:", out_file)
                    printed = True

            dt = time.time() - t_ens0
            rate = n_done / dt if dt > 0 else 0

            print(f"  → {ens} done: {dt/60:.1f} min ({rate:.2f} files/s)")

    # --------------------------------------------------
    # single-member (WDA)
    # --------------------------------------------------
    else:

        filelist = sorted(
            glob.glob(os.path.join(
                exp_dir,
                f"sigma0_{exp}_*.nc"
            ))
        )

        print(f"{exp}: {len(filelist)} files")

        out_dir = os.path.join(MLD_ROOT, exp)
        os.makedirs(out_dir, exist_ok=True)

        printed = False
        n_done = 0
        t_ens0 = time.time()

        for f in filelist:

            fname = os.path.basename(f)
            out_name = fname.replace("sigma0", "MLD")
            out_file = os.path.join(out_dir, out_name)

            if os.path.exists(out_file):
                continue

            ds = xr.open_dataset(f)
            sig = ds["sigma0"]

            MLD = compute_MLD_sigma(sig)

            encoding = {
                "MLD": {
                    "zlib": True,
                    "complevel": 4,
                    "dtype": "float32"
                }
            }

            MLD.to_netcdf(out_file, encoding=encoding)

            ds.close()
            del ds, sig, MLD

            n_done += 1

            if not printed:
                print("MLD:", out_file)
                printed = True

        dt = time.time() - t_ens0
        rate = n_done / dt if dt > 0 else 0
        print(f"  → single done: {dt/60:.1f} min ({rate:.2f} files/s)")


# --------------------------------------------------
# GLOBAL TIME
# --------------------------------------------------
dt_global = time.time() - t_global0
print(f"\nTOTAL elapsed: {dt_global/3600:.2f} hr")


===== LE =====
LE 0: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/0/MLD_LE_0_1955.nc
  → 0 done: 0.1 min (14.37 files/s)
LE 1: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/1/MLD_LE_1_1955.nc
  → 1 done: 0.1 min (14.24 files/s)
LE 10: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/10/MLD_LE_10_1955.nc
  → 10 done: 0.1 min (14.20 files/s)
LE 11: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/11/MLD_LE_11_1955.nc
  → 11 done: 0.1 min (15.12 files/s)
LE 12: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/12/MLD_LE_12_1955.nc
  → 12 done: 0.1 min (13.87 files/s)
LE 13: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/13/MLD_LE_13_1955.nc
  → 13 done: 0.1 min (13.39 files/s)
LE 14: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/14/MLD_LE_14_1955.nc
  → 14 done: 0.1 min (14.58 files/s)
LE 15: 66 files
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE/15/MLD_LE_15_1955.nc
  → 15 done: 0.1 min (9.53 files/s)
LE 16: 66